# Data Preparation: Finetuning Datasets

Loads two datasets from the [UCI Machine Learning Repository](https://archive.ics.uci.edu/) — one for **classification** and one for **regression** — splits each 80/20 into train/score, and saves the results to Unity Catalog.

We use datasets with more than 100k samples to resemble real-world examples where GPU memory becomes a bottleneck. The datasets are used **as-is without any transformation** to demonstrate AI Runtime finetuning capabilities in Databricks on real datasets.

### Datasets

| Task | Dataset | Rows | Features | Target | UCI ID |
| --- | --- | --- | --- | --- | --- |
| Classification | [Dota2 Games Results](https://archive.ics.uci.edu/dataset/367/dota2+games+results) | 102,944 | 115 | `win` | 367 |
| Regression | [RT-IoT2022](https://archive.ics.uci.edu/dataset/942/rt-iot2022) | 123,117 | 83 | `flow_duration` | 942 |

### License & Attribution

Both datasets are provided under the [Creative Commons Attribution 4.0 International (CC BY 4.0)](https://creativecommons.org/licenses/by/4.0/) license. Under this license you are free to share and adapt the material for any purpose, including commercial, as long as appropriate credit is given.

**Dota2 Games Results** (2016)\
Creator: Stephen Tridgell. DOI: [10.24432/C5W593](https://doi.org/10.24432/C5W593). Source: [UCI ML Repository](https://archive.ics.uci.edu/dataset/367/dota2+games+results).

**RT-IoT2022** (2023)\
Creators: B. S. Sharmila, Rohini Nagapadma. DOI: [10.24432/C5P338](https://doi.org/10.24432/C5P338). Source: [UCI ML Repository](https://archive.ics.uci.edu/dataset/942/rt-iot2022).

**No modifications were made.** Both datasets are used as-is without any transformation.

**Disclaimer:** The licensed material is provided "as-is" and "as-available". The licensors make no warranties regarding the material. See [Section 5 of the CC BY 4.0 license](https://creativecommons.org/licenses/by/4.0/legalcode#s5) for the full disclaimer of warranties.

**Prerequisites:** `common/config.py` defines the target catalog. The data will be populated in its 'default' schema.

**Compute:** Serverless CPU, Standard environment version 6

## Configuration

In [0]:
%pip install ucimlrepo --quiet -q

In [0]:
dbutils.library.restartPython()

In [0]:
import importlib.util
import os
import sys

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from ucimlrepo import fetch_ucirepo

# ── Constants ──────────────────────────────────────────────────────────────────
SEED = 42

# Make the repo-level common/ package importable.
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
COMMON_PATH = os.path.join(REPO_ROOT, "common")
CONFIG_PATH = os.path.join(COMMON_PATH, "config.py")
SCHEMA = "default"

if not os.path.exists(CONFIG_PATH):
    raise FileNotFoundError(f"Could not find config.py at {CONFIG_PATH}")

spec = importlib.util.spec_from_file_location("project_config", CONFIG_PATH)
project_config = importlib.util.module_from_spec(spec)
spec.loader.exec_module(project_config)
CATALOG = project_config.CATALOG

spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SCHEMA}")
spark.sql(f"USE SCHEMA {SCHEMA}")

print(f"Catalog/schema: {CATALOG}.{SCHEMA}")

## Classification dataset: Dota2 Games Results

In [0]:
# --- Classification: Dota2 Games Results (UCI id=367) ---
dataset = fetch_ucirepo(id=367)
df = pd.concat([dataset.data.features, dataset.data.targets], axis=1)

print(f"Dataset: {dataset.metadata['name']}")
print(f"Shape: {df.shape}")
print(f"Rows: {df.shape[0]:,}, Columns: {df.shape[1]}")
print(f"Target: 'win'  |  Values: {sorted(df['win'].unique())}")

# Split 80/20
df_train, df_score = train_test_split(df, test_size=0.2, random_state=SEED)

print(f"\nTrain: {df_train.shape[0]:,} rows")
print(f"Score: {df_score.shape[0]:,} rows")

# Save to Unity Catalog
TABLE_TRAIN = "finetuning_classification_train"
TABLE_SCORE = "finetuning_classification_score"

spark.createDataFrame(df_train).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{CATALOG}.{SCHEMA}.{TABLE_TRAIN}")
print(f"\n✓ Saved {df_train.shape[0]:,} rows to {CATALOG}.{SCHEMA}.{TABLE_TRAIN}")

spark.createDataFrame(df_score).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{CATALOG}.{SCHEMA}.{TABLE_SCORE}")
print(f"✓ Saved {df_score.shape[0]:,} rows to {CATALOG}.{SCHEMA}.{TABLE_SCORE}")

## Regression dataset: RT-IoT2022

In [0]:
# --- Regression: RT-IoT2022 (UCI id=942) ---
# The dataset's default target is Attack_type (classification).
# For regression we use `flow_duration` — the duration of each network flow.
REGRESSION_TARGET = "flow_duration"

dataset = fetch_ucirepo(id=942)
df = pd.concat([dataset.data.features, dataset.data.targets], axis=1)

# Convert object columns to str for Spark/Arrow compatibility
df = df.astype({c: str for c in df.select_dtypes("object").columns})

print(f"Dataset: {dataset.metadata['name']}")
print(f"Shape: {df.shape}")
print(f"Rows: {df.shape[0]:,}, Columns: {df.shape[1]}")
print(f"Regression target: '{REGRESSION_TARGET}'")
print(f"Target stats:\n{df[REGRESSION_TARGET].describe()}")

# Split 80/20
df_train, df_score = train_test_split(df, test_size=0.2, random_state=SEED)

print(f"\nTrain: {df_train.shape[0]:,} rows")
print(f"Score: {df_score.shape[0]:,} rows")

# Save to Unity Catalog
TABLE_TRAIN = "finetuning_regression_train"
TABLE_SCORE = "finetuning_regression_score"

spark.createDataFrame(df_train).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{CATALOG}.{SCHEMA}.{TABLE_TRAIN}")
print(f"\n✓ Saved {df_train.shape[0]:,} rows to {CATALOG}.{SCHEMA}.{TABLE_TRAIN}")

spark.createDataFrame(df_score).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{CATALOG}.{SCHEMA}.{TABLE_SCORE}")
print(f"✓ Saved {df_score.shape[0]:,} rows to {CATALOG}.{SCHEMA}.{TABLE_SCORE}")

## Create Volumes to store finetuned models

This is not needed as MLFlow would handle the finetuned checkpoints, but we are storing them too in a volume to make explicit the reusability of finetuned models. 

In [0]:
# --- Create volumes for finetuned model artifacts ---
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.finetuned_classification_models")
print(f"✓ Volume: {CATALOG}.{SCHEMA}.finetuned_classification_models")

spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.finetuned_regression_models")
print(f"✓ Volume: {CATALOG}.{SCHEMA}.finetuned_regression_models")